# Notebook 7b: Variantes de Hiperparámetros (UserKNN + SVD) - METODOLOGÍA CORRECTA

**IMPORTANTE:** Este notebook REEMPLAZA `calculate_ranking_variants.py`

Calcula métricas completas para:
- UserKNN con k ∈ {20, 30, 40}
- SVD con factores ∈ {20, 50, 100}
- Fracciones: 25%, 50%, 75%

**METODOLOGÍA:** train_test_split INTERNO (como Notebook 7)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from surprise import Dataset, Reader, KNNBasic, SVD
from surprise.model_selection import train_test_split
from codecarbon import EmissionsTracker
from collections import defaultdict
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

## Configuración de rutas

In [2]:
DATA_PATH = Path("../data/processed")
SUBSAMPLED_PATH = DATA_PATH / "subsampled_informed"
RESULTS_PATH = Path("../results")
RESULTS_PATH.mkdir(exist_ok=True, parents=True)

print(f" Directorio de datos: {DATA_PATH}")
print(f" Directorio subsampling: {SUBSAMPLED_PATH}")
print(f" Directorio resultados: {RESULTS_PATH}")

 Directorio de datos: ..\data\processed
 Directorio subsampling: ..\data\processed\subsampled_informed
 Directorio resultados: ..\results


## Funciones auxiliares (idénticas al Notebook 7)

In [3]:
def get_top_k(predictions, k=10):
    """Extrae top-K predicciones por usuario"""
    user_ratings = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        user_ratings[uid].append((iid, est))
    for uid, ratings in user_ratings.items():
        ratings.sort(key=lambda x: x[1], reverse=True)
        user_ratings[uid] = [iid for (iid, _) in ratings[:k]]
    return user_ratings


def precision_recall_at_k(predictions, k=10, threshold=3.5):
    """Calcula Precision@K y Recall@K (misma función del Notebook 7)"""
    top_k = get_top_k(predictions, k)
    user_metrics = []

    for uid, user_predictions in top_k.items():
        true_items = [iid for (u, iid, true, _, _) in predictions 
                     if u == uid and true >= threshold]

        if len(user_predictions) == 0:
            continue

        precision = len(set(user_predictions) & set(true_items)) / k if k > 0 else 0
        recall = (len(set(user_predictions) & set(true_items)) / len(true_items) 
                 if len(true_items) > 0 else 0)

        user_metrics.append((precision, recall))

    if not user_metrics:
        return 0, 0

    precision_mean = np.mean([x[0] for x in user_metrics])
    recall_mean = np.mean([x[1] for x in user_metrics])

    return precision_mean, recall_mean

## Función principal de entrenamiento y evaluación

In [4]:
def train_and_evaluate_model(file_path, model, model_name, fraction):
    """
    Entrena y evalúa un modelo usando METODOLOGÍA CORRECTA:
    - Carga subset completo
    - train_test_split INTERNO (80/20)
    - Evalúa sobre testset del MISMO subset
    """
    print(f"\n{'='*80}")
    print(f" {model_name} - Fracción {fraction}%")
    print(f"{'='*80}")
    
    df = pd.read_csv(file_path)
    print(f"   Cargado: {len(df):,} ratings")
    print(f"   Usuarios: {df['userId'].nunique():,}")
    print(f"   Películas: {df['movieId'].nunique():,}")
    
    reader = Reader(rating_scale=(df['rating'].min(), df['rating'].max()))
    data = Dataset.load_from_df(df[['userId', 'movieId', 'rating']], reader)
    
    trainset, testset = train_test_split(data, test_size=0.2, random_state=42)
    print(f"  Split: Train={trainset.n_ratings:,}, Test={len(testset):,}")
    
    print(f" Entrenando modelo...")
    tracker = EmissionsTracker(
        project_name=f"{model_name}_f{fraction}",
        log_level='error',
        save_to_file=False
    )
    tracker.start()
    
    model.fit(trainset)
    
    emissions = tracker.stop()
    
    print(f" Evaluando modelo...")
    predictions = model.test(testset)
    
    rmse_values = [abs(true - est) ** 2 for (_, _, true, est, _) in predictions]
    mae_values = [abs(true - est) for (_, _, true, est, _) in predictions]
    rmse = np.sqrt(np.mean(rmse_values))
    mae = np.mean(mae_values)
    
    precision, recall = precision_recall_at_k(predictions, k=10, threshold=3.5)
    
    result = {
        'method': model_name,
        'fraction': fraction,
        'rmse': rmse,
        'mae': mae,
        'precision_at_10': precision,
        'recall_at_10': recall,
        'co2_kg': emissions if emissions else 0.0,
        'n_train': trainset.n_ratings,
        'n_test': len(testset)
    }
    
    print(f"   Resultados:")
    print(f"   RMSE: {rmse:.4f}")
    print(f"   MAE: {mae:.4f}")
    print(f"   Precision@10: {precision:.4f}")
    print(f"   Recall@10: {recall:.4f}")
    print(f"   CO2: {emissions:.6f} kg" if emissions else "   CO2: No disponible")
    
    return result

## PARTE 1: Variantes UserKNN

In [5]:
print("\n" + "="*80)
print("PARTE 1: EVALUANDO VARIANTES UserKNN")
print("="*80)

k_values = [20, 30, 40]
fractions = [25, 50, 75]
userknn_results = []

for k_neighbors in k_values:
    for fraction in fractions:
        file_path = SUBSAMPLED_PATH / f"ratings_top_users_{fraction}.csv"
        
        if not file_path.exists():
            print(f" Archivo no encontrado: {file_path}")
            continue
        
        model = KNNBasic(
            k=k_neighbors,
            sim_options={'name': 'cosine', 'user_based': True},  # user_based=True
            verbose=False
        )
        
        model_name = f"UserKNN_k{k_neighbors}"
        
        result = train_and_evaluate_model(file_path, model, model_name, fraction)
        userknn_results.append(result)

df_userknn = pd.DataFrame(userknn_results)
output_file = RESULTS_PATH / "userknn_variants_metrics.csv"
df_userknn.to_csv(output_file, index=False)

print(f"\n Resultados UserKNN guardados en: {output_file}")
print(f"\n Resumen UserKNN:")
print(df_userknn.to_string(index=False))


PARTE 1: EVALUANDO VARIANTES UserKNN

 UserKNN_k20 - Fracción 25%
   Cargado: 640,906 ratings
   Usuarios: 1,510
   Películas: 3,656


[codecarbon WARNING @ 18:23:42] Multiple instances of codecarbon are allowed to run at the same time.


  Split: Train=512,724, Test=128,182
 Entrenando modelo...
 Evaluando modelo...
   Resultados:
   RMSE: 0.9719
   MAE: 0.7632
   Precision@10: 0.8332
   Recall@10: 0.2247
   CO2: 0.000156 kg

 UserKNN_k20 - Fracción 50%
   Cargado: 854,612 ratings
   Usuarios: 3,020
   Películas: 3,671
  Split: Train=683,689, Test=170,923
 Entrenando modelo...
 Evaluando modelo...
   Resultados:
   RMSE: 0.9800
   MAE: 0.7692
   Precision@10: 0.8010
   Recall@10: 0.3521
   CO2: 0.000317 kg

 UserKNN_k20 - Fracción 75%
   Cargado: 954,369 ratings
   Usuarios: 4,530
   Películas: 3,691
  Split: Train=763,495, Test=190,874
 Entrenando modelo...
 Evaluando modelo...
   Resultados:
   RMSE: 0.9829
   MAE: 0.7740
   Precision@10: 0.7527
   Recall@10: 0.5100
   CO2: 0.000441 kg

 UserKNN_k30 - Fracción 25%
   Cargado: 640,906 ratings
   Usuarios: 1,510
   Películas: 3,656
  Split: Train=512,724, Test=128,182
 Entrenando modelo...
 Evaluando modelo...
   Resultados:
   RMSE: 0.9670
   MAE: 0.7602
   Precision@

## PARTE 2: Variantes SVD

In [6]:
print("\n" + "="*80)
print("PARTE 2: EVALUANDO VARIANTES SVD")
print("="*80)

n_factors_values = [20, 50, 100]
fractions = [25, 50, 75]
N_EPOCHS = 20
LEARNING_RATE = 0.005
svd_results = []

for n_factors in n_factors_values:
    for fraction in fractions:
        file_path = SUBSAMPLED_PATH / f"ratings_top_items_{fraction}.csv"
        
        if not file_path.exists():
            print(f"  Archivo no encontrado: {file_path}")
            continue
        
        model = SVD(
            n_factors=n_factors,
            n_epochs=N_EPOCHS,
            lr_all=LEARNING_RATE,
            random_state=42,
            verbose=False
        )
        
        model_name = f"SVD_f{n_factors}"
        
        result = train_and_evaluate_model(file_path, model, model_name, fraction)
        svd_results.append(result)

df_svd = pd.DataFrame(svd_results)
output_file = RESULTS_PATH / "svd_variants_metrics.csv"
df_svd.to_csv(output_file, index=False)

print(f"\n Resultados SVD guardados en: {output_file}")
print(f"\n Resumen SVD:")
print(df_svd.to_string(index=False))


PARTE 2: EVALUANDO VARIANTES SVD

 SVD_f20 - Fracción 25%
   Cargado: 723,075 ratings
   Usuarios: 6,040
   Películas: 926
  Split: Train=578,460, Test=144,615
 Entrenando modelo...
 Evaluando modelo...
   Resultados:
   RMSE: 0.8606
   MAE: 0.6767
   Precision@10: 0.6483
   Recall@10: 0.6867
   CO2: 0.000033 kg

 SVD_f20 - Fracción 50%
   Cargado: 923,889 ratings
   Usuarios: 6,040
   Películas: 1,853
  Split: Train=739,111, Test=184,778
 Entrenando modelo...
 Evaluando modelo...
   Resultados:
   RMSE: 0.8686
   MAE: 0.6835
   Precision@10: 0.6761
   Recall@10: 0.6454
   CO2: 0.000045 kg

 SVD_f20 - Fracción 75%
   Cargado: 988,678 ratings
   Usuarios: 6,040
   Películas: 2,779
  Split: Train=790,942, Test=197,736
 Entrenando modelo...
 Evaluando modelo...
   Resultados:
   RMSE: 0.8687
   MAE: 0.6824
   Precision@10: 0.6806
   Recall@10: 0.6355
   CO2: 0.000045 kg

 SVD_f50 - Fracción 25%
   Cargado: 723,075 ratings
   Usuarios: 6,040
   Películas: 926
  Split: Train=578,460, Test=

## PARTE 3: Combinar todos los resultados

In [7]:
print("\n" + "="*80)
print("PARTE 3: COMBINANDO TODOS LOS RESULTADOS")
print("="*80)

all_variants = pd.concat([df_userknn, df_svd], ignore_index=True)

combined_file = RESULTS_PATH / "userknn_svd_variants_metrics.csv"
all_variants.to_csv(combined_file, index=False)

print(f"\n Resultados combinados guardados en: {combined_file}")
print(f"\n Resumen COMPLETO (UserKNN + SVD):")
print(all_variants.to_string(index=False))


PARTE 3: COMBINANDO TODOS LOS RESULTADOS

 Resultados combinados guardados en: ..\results\userknn_svd_variants_metrics.csv

 Resumen COMPLETO (UserKNN + SVD):
     method  fraction     rmse      mae  precision_at_10  recall_at_10   co2_kg  n_train  n_test
UserKNN_k20        25 0.971863 0.763197         0.833245      0.224659 0.000156   512724  128182
UserKNN_k20        50 0.980028 0.769225         0.801026      0.352096 0.000317   683689  170923
UserKNN_k20        75 0.982926 0.774049         0.752693      0.509996 0.000441   763495  190874
UserKNN_k30        25 0.967013 0.760227         0.834636      0.224996 0.000137   512724  128182
UserKNN_k30        50 0.973912 0.764711         0.808510      0.355261 0.000307   683689  170923
UserKNN_k30        75 0.975406 0.767897         0.758675      0.513606 0.000431   763495  190874
UserKNN_k40        25 0.964276 0.758632         0.836689      0.225584 0.000151   512724  128182
UserKNN_k40        50 0.970731 0.762418         0.813675      0.

## PARTE 4: Verificación de coherencia

In [8]:
print("\n" + "="*80)
print(" VERIFICACIÓN DE COHERENCIA")
print("="*80)

print(f"\n Estadísticas globales:")
print(f"   Total configuraciones: {len(all_variants)}")
print(f"   RMSE promedio: {all_variants['rmse'].mean():.4f}")
print(f"   MAE promedio: {all_variants['mae'].mean():.4f}")
print(f"   Precision@10 promedio: {all_variants['precision_at_10'].mean():.4f}")
print(f"   Recall@10 promedio: {all_variants['recall_at_10'].mean():.4f}")
print(f"   CO2 total: {all_variants['co2_kg'].sum():.6f} kg")

p_mean = all_variants['precision_at_10'].mean()
r_mean = all_variants['recall_at_10'].mean()

print(f"\n Verificación de valores:")

if p_mean < 0.1:
    print(f"     ADVERTENCIA: Precision@10 baja ({p_mean:.4f})")
    print(f"       Esperado: > 0.2")
elif p_mean > 0.2:
    print(f"   Precision@10 en rango razonable ({p_mean:.4f})")
else:
    print(f"   Precision@10 marginal ({p_mean:.4f})")

if r_mean < 0.1:
    print(f"    ADVERTENCIA: Recall@10 bajo ({r_mean:.4f})")
    print(f"       Esperado: > 0.2")
elif r_mean > 0.2:
    print(f"   Recall@10 en rango razonable ({r_mean:.4f})")
else:
    print(f"   Recall@10 marginal ({r_mean:.4f})")

print(f"\n Comparación con valores de referencia (informed_users):")
print(f"   Referencia informed_users_25: P@10=0.854, R@10=0.231")
print(f"   UserKNN_k40 (25%): P@10={df_userknn[df_userknn['method']=='UserKNN_k40']['precision_at_10'].iloc[0]:.3f}, R@10={df_userknn[df_userknn['method']=='UserKNN_k40']['recall_at_10'].iloc[0]:.3f}")

print(f"\n{'='*80}")
print(" NOTEBOOK 7b COMPLETADO EXITOSAMENTE")
print("="*80)
print("\nArchivos generados:")
print(f"   1. {RESULTS_PATH / 'userknn_variants_metrics.csv'}")
print(f"   2. {RESULTS_PATH / 'svd_variants_metrics.csv'}")
print(f"   3. {RESULTS_PATH / 'userknn_svd_variants_metrics.csv'} (combinado)")
print("\nSiguiente paso: Actualizar Tabla 3.1 en el TFM")


 VERIFICACIÓN DE COHERENCIA

 Estadísticas globales:
   Total configuraciones: 18
   RMSE promedio: 0.9194
   MAE promedio: 0.7224
   Precision@10 promedio: 0.7345
   Recall@10 promedio: 0.5101
   CO2 total: 0.003209 kg

 Verificación de valores:
   Precision@10 en rango razonable (0.7345)
   Recall@10 en rango razonable (0.5101)

 Comparación con valores de referencia (informed_users):
   Referencia informed_users_25: P@10=0.854, R@10=0.231
   UserKNN_k40 (25%): P@10=0.837, R@10=0.226

 NOTEBOOK 7b COMPLETADO EXITOSAMENTE

Archivos generados:
   1. ..\results\userknn_variants_metrics.csv
   2. ..\results\svd_variants_metrics.csv
   3. ..\results\userknn_svd_variants_metrics.csv (combinado)

Siguiente paso: Actualizar Tabla 3.1 en el TFM
